In [ ]:
import os
from crewai import Agent, Task, Crew, Process, LLM
from langchain_groq import ChatGroq
from exa_py import Exa
from datetime import datetime



In [ ]:
# Set up API keys
os.environ["GROQ_API_KEY"] = input("Enter your GROQ API Key: ")
os.environ["EXA_API_KEY"] = input("Enter your EXA API Key: ")


exa_client = Exa(api_key=os.environ["EXA_API_KEY"])

In [ ]:
# # Initialize Groq Llama 3 model

llm = LLM(
    model="llama-3.3-70b-versatile",
    api_key=os.environ["GROQ_API_KEY"],
    base_url="https://api.groq.com/openai/v1",
)

In [ ]:
# --- Threat Intelligence Gathering Agent ---
def fetch_cybersecurity_threats(query):
    result = exa_client.search_and_contents(query, summary=True)

    if result.results:
        threat_list = []
        for item in result.results:
            threat_list.append({
                "title": item.title if hasattr(item, "title") else "No Title",
                "url": item.url if hasattr(item, "url") else "#",
                "published_date": item.published_date if hasattr(item, "published_date") else "Unknown Date",
                "summary": item.summary if hasattr(item, "summary") else "No Summary",
            })
        return threat_list
    else:
        return []  # Return empty list if no results found

threat_analyst = Agent(
    role="Cybersecurity Threat Intelligence Analyst",
    goal="Gather real-time cybersecurity threat intelligence.",
    backstory="You're an expert in cybersecurity, tracking emerging threats, malware campaigns, and hacking incidents.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=False,
)

threat_analysis_task = Task(
    description=f"Use EXA API to retrieve the latest cybersecurity threats. Provide a summary of the top threats.",
    expected_output="A structured list of recent cybersecurity threats, including malware trends and cyberattacks.",
    agent=threat_analyst,
    callback=lambda inputs: fetch_cybersecurity_threats("Latest cybersecurity threats 2024"),
)



In [ ]:
# --- Vulnerability Researcher Agent ---
def fetch_latest_cves():
    cve_query = "Latest CVEs and security vulnerabilities 2024"
    result = exa_client.search_and_contents(cve_query, summary=True)

    if result.results:
        cve_list = []
        for item in result.results[:5]:  # Fetch top 5 vulnerabilities
            cve_list.append({
                "title": item.title if hasattr(item, "title") else "No Title",
                "url": item.url if hasattr(item, "url") else "#",
                "published_date": item.published_date if hasattr(item, "published_date") else "Unknown Date",
                "summary": item.summary if hasattr(item, "summary") else "No Summary",
            })
        return cve_list
    else:
        return []  # Return empty list if no results found

vulnerability_researcher = Agent(
    role="Vulnerability Researcher",
    goal="Identify the latest software vulnerabilities and security flaws.",
    backstory="You're a cybersecurity researcher specializing in vulnerability analysis and threat mitigation.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=False,
)

vulnerability_research_task = Task(
    description="Fetch and analyze the latest security vulnerabilities (CVEs).",
    expected_output="A structured list of newly discovered CVEs and their impact.",
    agent=vulnerability_researcher,
    callback=lambda inputs: fetch_latest_cves(),
)


In [ ]:
# --- Incident Response Advisor Agent ---
incident_response_advisor = Agent(
    role="Incident Response Advisor",
    goal="Provide mitigation strategies for detected threats and vulnerabilities.",
    backstory="You specialize in cybersecurity defense strategies, helping organizations respond to security incidents effectively.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=False,
)

incident_response_task = Task(
    description="Analyze cybersecurity threats and vulnerabilities to suggest mitigation strategies.",
    expected_output="A list of recommended defensive actions against active threats.",
    agent=incident_response_advisor,
    context=[threat_analysis_task, vulnerability_research_task],
)



In [ ]:
# --- Cybersecurity Report Writer Agent ---
cybersecurity_writer = Agent(
    role="Cybersecurity Report Writer",
    goal="Generate a structured cybersecurity threat report based on collected intelligence.",
    backstory="You're a leading cybersecurity analyst with years of experience summarizing security reports and providing executive-level insights.",
    verbose=True,
    allow_delegation=False,
    llm=llm,
    max_iter=5,
    memory=False,
)

write_threat_report_task = Task(
    description="Summarize the cybersecurity threat intelligence, vulnerabilities, and response strategies into a report.",
    expected_output="A comprehensive cybersecurity intelligence report with key threats, vulnerabilities, and recommendations.",
    agent=cybersecurity_writer,
    context=[threat_analysis_task, vulnerability_research_task, incident_response_task],
)



In [ ]:
# --- Create and Run the Crew ---
crew = Crew(
    agents=[threat_analyst, vulnerability_researcher, incident_response_advisor, cybersecurity_writer],
    tasks=[threat_analysis_task, vulnerability_research_task, incident_response_task, write_threat_report_task],
    verbose=True,
    process=Process.sequential,
    full_output=True,
    share_crew=False,
    manager_llm=llm,
    max_iter=15,
)

results = crew.kickoff()

In [ ]:
from IPython.display import display, Markdown

# Convert the dictionary to a Markdown string before displaying
# display(Markdown(results['final_output']))
display(Markdown(results.raw))